# Exploracao - Datasus SIA e SIH

Notebook para explorar a camada canonica `data/processed` com consultas DuckDB prontas.

In [ ]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_GLOB = (PROJECT_ROOT / 'data' / 'processed' / '**' / '*.parquet').as_posix()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

def run_sql(sql: str):
    with duckdb.connect(database=':memory:') as con:
        con.execute(f"CREATE OR REPLACE VIEW processed AS SELECT * FROM read_parquet('{PROCESSED_GLOB}', union_by_name=true)")
        return con.execute(sql).df()

if not list(PROCESSED_DIR.rglob('*.parquet')):
    print(f'Nenhum parquet em {PROCESSED_DIR}. Execute: python -m src.data.transform')
else:
    print(f'Camada processed encontrada em {PROCESSED_DIR}.')

In [ ]:
# Visao rapida: amostra da base canonica
run_sql("""
SELECT
  row_id,
  sistema,
  competencia_ano_mes,
  cod_procedimento,
  cid_principal,
  custo_total
FROM processed
LIMIT 10
""")

In [ ]:
# Serie mensal de volume e custo
run_sql("""
SELECT
  competencia_ano_mes,
  COUNT(*) AS total_registros,
  ROUND(SUM(custo_total), 2) AS custo_total_mes,
  ROUND(AVG(custo_total), 2) AS custo_medio_mes
FROM processed
GROUP BY competencia_ano_mes
ORDER BY competencia_ano_mes
""")

In [ ]:
# Top procedimentos por quantidade e gasto
run_sql("""
SELECT
  cod_procedimento,
  COUNT(*) AS total_registros,
  ROUND(SUM(custo_total), 2) AS custo_total,
  ROUND(AVG(custo_total), 2) AS custo_medio
FROM processed
WHERE cod_procedimento IS NOT NULL
GROUP BY cod_procedimento
ORDER BY custo_total DESC
LIMIT 20
""")

In [ ]:
# Distribuicao por sexo e faixa etaria aproximada
run_sql("""
SELECT
  COALESCE(sexo_paciente, 'NI') AS sexo_paciente,
  CASE
    WHEN idade_paciente IS NULL THEN 'idade_desconhecida'
    WHEN idade_paciente < 18 THEN '00-17'
    WHEN idade_paciente < 40 THEN '18-39'
    WHEN idade_paciente < 60 THEN '40-59'
    ELSE '60+'
  END AS faixa_etaria,
  COUNT(*) AS total_registros,
  ROUND(SUM(custo_total), 2) AS custo_total
FROM processed
GROUP BY 1, 2
ORDER BY sexo_paciente, faixa_etaria
""")

In [ ]:
# Qualidade: nulos em colunas-chave
run_sql("""
SELECT
  COUNT(*) AS total_linhas,
  SUM(CASE WHEN cid_principal IS NULL OR TRIM(cid_principal) = '' THEN 1 ELSE 0 END) AS nulos_cid_principal,
  SUM(CASE WHEN cod_procedimento IS NULL OR TRIM(cod_procedimento) = '' THEN 1 ELSE 0 END) AS nulos_cod_procedimento,
  SUM(CASE WHEN custo_total IS NULL THEN 1 ELSE 0 END) AS nulos_custo_total,
  SUM(CASE WHEN competencia_ano_mes IS NULL THEN 1 ELSE 0 END) AS nulos_competencia
FROM processed
""")

In [ ]:
# Recorte parametrizado para exploracao rapida
competencia = 202204
limite = 20

run_sql(f"""
SELECT
  sistema,
  competencia_ano_mes,
  cod_munic_residencia,
  cod_procedimento,
  cid_principal,
  custo_total
FROM processed
WHERE competencia_ano_mes = {competencia}
ORDER BY custo_total DESC NULLS LAST
LIMIT {limite}
""")